# Quick look at the data

Before writing the project brief I wanted real numbers in front of me instead of guesses, so this notebook just loads the CSVs and pulls a few totals. No cleaning yet, that comes later.

Data: *The Complete Journey* by dunnhumby, downloaded from Kaggle into `data/raw/`. I'm leaving out `causal_data.csv` (the promotions file) for now because it's ~700 MB and I don't need it yet.

In [ ]:
import pandas as pd

raw = "../data/raw/"

tx = pd.read_csv(raw + "transaction_data.csv")
campaigns = pd.read_csv(raw + "campaign_table.csv")
campaign_desc = pd.read_csv(raw + "campaign_desc.csv")
demo = pd.read_csv(raw + "hh_demographic.csv")

# the original files mix UPPER and lower case column names, so I'm making them all lowercase
for df in [tx, campaigns, campaign_desc, demo]:
    df.columns = df.columns.str.lower()

print(tx.shape)
tx.head()

There's no real date column. `day` is a day number (day 1 = first day of the data) and `week_no` is the week number, so I'll use those for anything time-based.

## Sales

I'm splitting total sales into **households × trips per household × spend per trip**. If those three multiply back to the total, the numbers are consistent. It also means that later on I can say whether a change in sales came from more customers, more visits or bigger baskets.

In [ ]:
total_sales = tx["sales_value"].sum()
households = tx["household_key"].nunique()
trips = tx["basket_id"].nunique()      # one basket = one shopping trip

trips_per_household = trips / households
spend_per_trip = total_sales / trips

print("Days covered:", tx["day"].min(), "to", tx["day"].max())
print("Weeks covered:", tx["week_no"].min(), "to", tx["week_no"].max())
print("Stores:", tx["store_id"].nunique())
print(f"Total sales: ${total_sales:,.0f}")
print(f"Households: {households:,}")
print(f"Trips: {trips:,}")
print(f"Trips per household: {trips_per_household:.1f}")
print(f"Spend per trip: ${spend_per_trip:.2f}")

In [ ]:
# sanity check: does households x trips x spend give back total sales?
rebuilt = households * trips_per_household * spend_per_trip
print(f"${rebuilt:,.0f}", "matches" if round(rebuilt) == round(total_sales) else "DOESN'T MATCH")

## Discounts

In this version of the data, discounts are stored as negative numbers. Checking that before adding anything up.

In [ ]:
tx[["retail_disc", "coupon_disc"]].describe()

In [ ]:
# abs() turns the negatives into positive dollar amounts
retail_disc = tx["retail_disc"].abs().sum()
coupon_disc = tx["coupon_disc"].abs().sum()
any_discount = (tx["retail_disc"] != 0) | (tx["coupon_disc"] != 0)

print(f"Retail discounts: ${retail_disc:,.0f}")
print(f"Coupon discounts: ${coupon_disc:,.0f}")
print(f"Discounts as % of sales: {(retail_disc + coupon_disc) / total_sales:.1%}")
print(f"Purchase lines with some discount: {any_discount.mean():.0%}")

*(Write one or two lines here once you've seen the numbers: what share of sales went to discounts, and why that matters for the project.)*

## Campaigns and demographics

In [ ]:
reached = campaigns["household_key"].nunique()

print("Campaigns:", campaign_desc["campaign"].nunique())
print(campaign_desc["description"].value_counts().sort_index())
print(f"Households that got at least one campaign: {reached:,} of {households:,} ({reached / households:.0%})")
print(f"Households with demographic info: {demo['household_key'].nunique():,} of {households:,}")

## Things to remember for the brief

*(Fill in after running: time span, households, stores, how many households have demographics, how many never got a campaign.)*